In [1]:
import urllib.request
import os
from datetime import datetime

os.makedirs('vhi_data', exist_ok=True)

def download_vhi_data():
    for province_id in range(1, 28): # від 1 до 27 областей
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
        
        existing_files = [f for f in os.listdir('vhi_data') if f.startswith(f"vhi_province_{province_id}_")]
        
        if existing_files:
            print(f"Файл для області {province_id} вже існує. Пропускаємо.")
            continue
    
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"vhi_data/vhi_province_{province_id}_{timestamp}.csv"
        
        try:
            urllib.request.urlretrieve(url, filename)
            print(f"Завантажено: {filename}")
        except Exception as e:
            print(f"Помилка завантаження для області {province_id}: {e}")

download_vhi_data()

Завантажено: vhi_data/vhi_province_1_20260305_080910.csv
Завантажено: vhi_data/vhi_province_2_20260305_080914.csv
Завантажено: vhi_data/vhi_province_3_20260305_080916.csv
Завантажено: vhi_data/vhi_province_4_20260305_080918.csv
Завантажено: vhi_data/vhi_province_5_20260305_080920.csv
Завантажено: vhi_data/vhi_province_6_20260305_080921.csv
Завантажено: vhi_data/vhi_province_7_20260305_080922.csv
Завантажено: vhi_data/vhi_province_8_20260305_080924.csv
Завантажено: vhi_data/vhi_province_9_20260305_080925.csv
Завантажено: vhi_data/vhi_province_10_20260305_080926.csv
Завантажено: vhi_data/vhi_province_11_20260305_080928.csv
Завантажено: vhi_data/vhi_province_12_20260305_080929.csv
Завантажено: vhi_data/vhi_province_13_20260305_080930.csv
Завантажено: vhi_data/vhi_province_14_20260305_080931.csv
Завантажено: vhi_data/vhi_province_15_20260305_080932.csv
Завантажено: vhi_data/vhi_province_16_20260305_080934.csv
Завантажено: vhi_data/vhi_province_17_20260305_080935.csv
Завантажено: vhi_data/v

### Завдання 2: Зчитування та очищення даних
Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning. Додати стовпчики з назвою та індексом області. Реалізувати процедуру зміни індексів на українську абетку (1 - Вінницька).

In [4]:
import pandas as pd
import glob

# словник
province_mapping = {
    1: (22, 'Черкаська'), 2: (24, 'Чернігівська'), 3: (23, 'Чернівецька'), 
    4: (25, 'Крим'), 5: (3, 'Дніпропетровська'), 6: (4, 'Донецька'), 
    7: (8, 'Івано-Франківська'), 8: (19, 'Харківська'), 9: (20, 'Херсонська'), 
    10: (21, 'Хмельницька'), 11: (9, 'Київська'), 12: (26, 'м. Київ'), 
    13: (11, 'Кіровоградська'), 14: (12, 'Луганська'), 15: (13, 'Львівська'), 
    16: (14, 'Миколаївська'), 17: (15, 'Одеська'), 18: (16, 'Полтавська'), 
    19: (17, 'Рівненська'), 20: (18, 'Сумська'), 21: (19, 'Тернопільська'), 
    22: (7, 'Закарпатська'), 23: (1, 'Вінницька'), 24: (2, 'Волинська'), 
    25: (6, 'Запорізька'), 26: (5, 'Житомирська'), 27: (27, 'м. Севастополь')
}

def load_and_clean_vhi_data(folder_path='vhi_data'):
    files = glob.glob(f"{folder_path}/*.csv")
    df_list = []
    
    for file in files:
        # витягую старий індекс області з назви файлу
        filename = os.path.basename(file)
        province_id = int(filename.split('_')[2])
        
        # читаю файл
        headers = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty']
        df = pd.read_csv(file, header=1, names=headers)
        
        # чистка даних
        df = df.drop(columns=['empty']) # Видаляємо порожній останній стовпець
        df = df.dropna() # Видаляємо пропуски
        
        # видаляю рядки, де замість року написано HTML-тег
        df = df[~df['Year'].astype(str).str.contains("</pre>")]
        df = df[~df['Year'].astype(str).str.contains("<tt>")]
        
        df['Year'] = df['Year'].astype(int)
        df['Week'] = df['Week'].astype(int)
        
        # додаю нові стовпці за допомогою словника
        new_id, province_name = province_mapping.get(province_id, (None, None))
        df['Province_Index_New'] = new_id
        df['Province_Name'] = province_name
        
        df_list.append(df)
        
    # об'єдную всі таблиці в одну велику
    full_df = pd.concat(df_list, ignore_index=True)
    return full_df

# викликаю функцію та зберігаю результат у змінну df_vhi
df_vhi = load_and_clean_vhi_data()

# виводжу перші 5 рядків, щоб перевірити, як виглядає таблиця
display(df_vhi.head())

,Year,Week,SMN,SMT,VCI,TCI,VHI,Province_Index_New,Province_Name
0,1982,2,0.063,261.53,55.89,38.20,47.04,21,Хмельницька
1,1982,3,0.063,263.45,57.30,32.69,44.99,21,Хмельницька
2,1982,4,0.061,265.10,53.96,28.62,41.29,21,Хмельницька
3,1982,5,0.058,266.42,46.87,28.57,37.72,21,Хмельницька
4,1982,6,0.056,267.47,39.55,30.27,34.91,21,Хмельницька


### Завдання 3: Вибірки даних
Реалізувати процедури для формування вибірок:
1. Ряд VHI для області за вказаний рік.
2. Ряд VHI за вказаний діапазон років для вказаних областей.

In [5]:
def get_vhi_by_year(df, province_index, year):
    # шукаю збіг по індексу області та року
    result = df[(df['Province_Index_New'] == province_index) & (df['Year'] == year)]
    return result[['Year', 'Week', 'VHI', 'Province_Name']]

vhi_year_result = get_vhi_by_year(df_vhi, 1, 2020)
print("VHI для Вінницької області за 2020 рік:")
display(vhi_year_result.head())

VHI для Вінницької області за 2020 рік:


,Year,Week,VHI,Province_Name
33265,2020,1,39.29,Вінницька
33266,2020,2,39.08,Вінницька
33267,2020,3,38.94,Вінницька
33268,2020,4,39.47,Вінницька
33269,2020,5,40.11,Вінницька


In [6]:
def get_vhi_by_year_range(df, province_index, start_year, end_year):
    result = df[(df['Province_Index_New'] == province_index) & 
                (df['Year'] >= start_year) & 
                (df['Year'] <= end_year)]
    return result[['Year', 'Week', 'VHI', 'Province_Name']]

vhi_range_result = get_vhi_by_year_range(df_vhi, 1, 2015, 2017)
print("VHI для Вінницької області за 2015-2017 роки:")
display(vhi_range_result.head())

VHI для Вінницької області за 2015-2017 роки:


,Year,Week,VHI,Province_Name
33005,2015,1,40.57,Вінницька
33006,2015,2,41.06,Вінницька
33007,2015,3,42.11,Вінницька
33008,2015,4,40.71,Вінницька
33009,2015,5,40.35,Вінницька


### Завдання 4: Пошук екстремумів та статистики
Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани.

In [7]:
def get_vhi_stats(df, province_index, year):
    filtered_df = df[(df['Province_Index_New'] == province_index) & (df['Year'] == year)]
    
    # знаходжу правильні значення
    vhi_min = filtered_df['VHI'].min()
    vhi_max = filtered_df['VHI'].max()
    vhi_mean = filtered_df['VHI'].mean()
    vhi_median = filtered_df['VHI'].median()
    
    # створюю зручну маленьку таблицю для красивого виводу
    stats_df = pd.DataFrame({
        'Показник': ['Мінімум (min)', 'Максимум (max)', 'Середнє (mean)', 'Медіана (median)'],
        'Значення VHI': [vhi_min, vhi_max, vhi_mean, vhi_median]
    })
    
    return stats_df

print("Статистика VHI для Київської області за 2019 рік:")
stats_result = get_vhi_stats(df_vhi, 11, 2019)
display(stats_result)

Статистика VHI для Київської області за 2019 рік:


,Показник,Значення VHI
0,Мінімум (min),22.080000
1,Максимум (max),73.050000
2,Середнє (mean),44.900962
3,Медіана (median),41.185000
